In [33]:
import pandas as pd 
import numpy as np
from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass

## Предобработка данных

### Разметка сплитов

In [34]:
DATA_ROOT = Path("datasets/")

In [35]:
@dataclass 
class Config:
    seed: int = 42 
    n_folds: int = 5 
    val_splits: list[str] = None 

def load_meta():
    train_log = pd.read_csv(DATA_ROOT / "train_log.csv")
    test_log = pd.read_csv(DATA_ROOT / "test_log.csv")
    return train_log, test_log

def get_splits(train_log: pd.DataFrame, n_folds: int = 5):
    splits, folds = sorted(train_log["split"].unique()), []
    for i in range(n_folds): folds.append(splits[i::n_folds])
    return folds 

def get_fold_indices(train_log: pd.DataFrame, val_splits: list[str]):
    is_val = train_log["split"].isin(val_splits)
    train_idx = train_log.index[~is_val].to_numpy()
    val_idx = train_log.index[is_val].to_numpy()
    return train_idx, val_idx


In [36]:
train_log, test_log = load_meta()
folds = get_splits(train_log, n_folds=5)
print('folds:', folds)


folds: [['split_01', 'split_06', 'split_11', 'split_16'], ['split_02', 'split_07', 'split_12', 'split_17'], ['split_03', 'split_08', 'split_13', 'split_18'], ['split_04', 'split_09', 'split_14', 'split_19'], ['split_05', 'split_10', 'split_15', 'split_20']]


### Загрузка кривых по нужным splits

In [37]:
class Curver:
    def load_lightcurves_for_splits(self, splits: list[str], kind: str) -> pd.DataFrame:
        parts = []
        for split in splits:
            fname = 'train_full_lightcurves.csv' if kind == "train" else "test_full_lightcurves.csv"
            path = DATA_ROOT / split / fname
            df = pd.read_csv(path)
            
            df["splits"] = split 
            parts.append(df)

        lc = pd.concat(parts, ignore_index=True)
        return lc 

    def clean_lightcurves(self, lc: pd.DataFrame) -> pd.DataFrame:
        lc = lc.copy()
        lc = lc[lc["Flux"].notna()]
        lc["Time (MJD)"] = lc["Time (MJD)"].astype(float)
        lc["Flux"] = lc["Flux"].astype(float)
        lc["Flux_err"] = lc["Flux_err"].astype(float)
        return lc 

    def add_time_features(self, lc: pd.DataFrame) -> pd.DataFrame:
        lc = lc.copy()
        lc["t0"] = lc.groupby("object_id")["Time (MJD)"].transform("min")
        lc["dt"] = lc["Time (MJD)"] - lc["t0"]
        return lc 

In [38]:
if __name__ == "__main__":
    curve = Curver()

    val_splits = folds[0]
    train_idx, val_idx = get_fold_indices(train_log, val_splits)

    # мета-таблицы
    train_meta = train_log.iloc[train_idx].copy()
    val_meta = train_log.iloc[val_idx].copy()

    # lightcurves
    train_lc = curve.load_lightcurves_for_splits(val_splits, kind="train")
    train_splits = [s for s in train_log["split"].unique() if s not in val_splits]
    train_lc = curve.load_lightcurves_for_splits(train_splits, kind="train")
    val_lc = curve.load_lightcurves_for_splits(val_splits, kind="train")

    train_lc = curve.add_time_features(curve.clean_lightcurves(train_lc))
    val_lc = curve.add_time_features(curve.clean_lightcurves(val_lc))


### Генерация полного набора фич

In [39]:
FILTERS = ['u', 'g', 'r', 'i', 'z', 'y']
TIME_BINS = [0, 30, 90, 180, 360, 720, 1440, 2880, 10000]


In [40]:
class Features:
    def _safe_stats(self, x: pd.Series):
        if x.empty:
            return {
                "count": 0, "mean": np.nan, "std": np.nan, "min": np.nan,
                "max": np.nan, "median": np.nan, "p10": np.nan, "p90": np.nan,
                "mad": np.nan, "iqr": np.nan, "skew": np.nan, "kurt": np.nan,
                "amp": np.nan,
            }
        med = x.median()
        mad = (x - med).abs().median()
        q25 = x.quantile(0.25)
        q75 = x.quantile(0.75)
        return {
            "count": x.count(),
            "mean": x.mean(),
            "std": x.std(ddof=0),
            "min": x.min(),
            "max": x.max(),
            "median": med,
            "p10": x.quantile(0.10),
            "p90": x.quantile(0.90),
            "mad": mad,
            "iqr": q75 - q25,
            "skew": x.skew(),
            "kurt": x.kurtosis(),
            "amp": x.max() - x.min(),
        }

    def _weighted_stats(self, x: np.ndarray, err: np.ndarray):
        mask = np.isfinite(x) & np.isfinite(err) & (err > 0)
        if not np.any(mask):
            return {"wmean": np.nan, "wstd": np.nan}
        x = x[mask]
        err = err[mask]
        w = 1.0 / (err ** 2 + 1e-6)
        wsum = np.sum(w)
        wmean = np.sum(w * x) / wsum
        wvar = np.sum(w * (x - wmean) ** 2) / wsum
        return {"wmean": wmean, "wstd": np.sqrt(wvar)}

    def _slope(self, dt: np.ndarray, flux: np.ndarray):
        if len(dt) < 2:
            return np.nan
        x = dt - dt.mean()
        y = flux - flux.mean()
        denom = np.sum(x * x)
        if denom == 0:
            return np.nan
        return np.sum(x * y) / denom

    def _max_rates(self, dt: np.ndarray, flux: np.ndarray):
        if len(dt) < 2:
            return np.nan, np.nan

        idx = np.argsort(dt)
        dt_sorted = dt[idx]
        flux_sorted = flux[idx]
        ddt = np.diff(dt_sorted)
        dflux = np.diff(flux_sorted)
        mask = ddt > 0
        if not np.any(mask):
            return np.nan, np.nan
        rates = dflux[mask] / ddt[mask]
        return np.max(rates), np.min(rates)

    def _cadence_stats(self, dt: np.ndarray):
        if len(dt) < 2:
            return {
                "span": 0.0,
                "gap_mean": np.nan,
                "gap_median": np.nan,
                "gap_max": np.nan,
                "gap_p90": np.nan,
            }
        dt_sorted = np.sort(dt)
        gaps = np.diff(dt_sorted)
        return {
            "span": dt_sorted[-1] - dt_sorted[0],
            "gap_mean": gaps.mean(),
            "gap_median": np.median(gaps),
            "gap_max": gaps.max(),
            "gap_p90": np.quantile(gaps, 0.9),
        }

    def _bin_stats(self, dt: np.ndarray, flux: np.ndarray):
        out = {}
        for i in range(len(TIME_BINS) - 1):
            lo = TIME_BINS[i]
            hi = TIME_BINS[i + 1]
            mask = (dt >= lo) & (dt < hi)
            key = f"bin_{int(lo)}_{int(hi)}"
            if not np.any(mask):
                out[f"{key}_count"] = 0
                out[f"{key}_mean"] = np.nan
                out[f"{key}_std"] = np.nan
                out[f"{key}_min"] = np.nan
                out[f"{key}_max"] = np.nan
            else:
                x = flux[mask]
                out[f"{key}_count"] = int(x.size)
                out[f"{key}_mean"] = float(np.mean(x))
                out[f"{key}_std"] = float(np.std(x))
                out[f"{key}_min"] = float(np.min(x))
                out[f"{key}_max"] = float(np.max(x))
        return out

    def _decay_fit(self, dt: np.ndarray, flux: np.ndarray, t_peak: float):
        mask = dt >= t_peak
        dt_post = dt[mask]
        flux_post = flux[mask]
        pos = flux_post > 0
        if np.sum(pos) < 3:
            return {"decay_slope": np.nan, "decay_r2": np.nan, "decay_n": int(np.sum(pos))}
        x = dt_post[pos] - t_peak
        y = np.log(flux_post[pos])
        x = x - x.mean()
        y = y - y.mean()
        denom = np.sum(x * x)
        if denom == 0:
            return {"decay_slope": np.nan, "decay_r2": np.nan, "decay_n": int(np.sum(pos))}
        slope = np.sum(x * y) / denom
        y_hat = slope * x
        ss_res = np.sum((y - y_hat) ** 2)
        ss_tot = np.sum(y ** 2)
        r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
        return {"decay_slope": slope, "decay_r2": r2, "decay_n": int(np.sum(pos))}


    def _shape_metrics(self, dt: np.ndarray, flux: np.ndarray):
        if len(dt) < 3:
            return {"asymmetry": np.nan, "smoothness": np.nan, "slope_var": np.nan}
        idx = np.argsort(dt)
        t = dt[idx]
        f = flux[idx]
        # asymmetry via rise/decay around peak
        peak_idx = int(np.nanargmax(f))
        t_peak = t[peak_idx]
        t_start = t[0]
        t_end = t[-1]
        t_rise = t_peak - t_start
        t_decay = t_end - t_peak
        asym = (t_decay - t_rise) / (t_decay + t_rise) if (t_decay + t_rise) > 0 else np.nan
        # smoothness: std of successive diff
        dflux = np.diff(f)
        smooth = np.std(dflux)
        # slope variance using dflux/dt
        dt_safe = np.diff(t)
        mask = dt_safe > 0
        if np.any(mask):
            slopes = dflux[mask] / dt_safe[mask]
            slope_var = np.var(slopes)
        else:
            slope_var = np.nan
        return {"asymmetry": asym, "smoothness": smooth, "slope_var": slope_var}
    def build_features(self, lc: pd.DataFrame) -> pd.DataFrame:
        features = []
        for obj_id, grp in lc.groupby("object_id"):
            row = {"object_id": obj_id}

            dt = grp["dt"].to_numpy()
            flux = grp["Flux"].to_numpy()
            ferr = grp["Flux_err"].to_numpy()

            row.update({f"cadence_{k}": v for k, v in self._cadence_stats(dt).items()})
            row["flux_slope_all"] = self._slope(dt, flux)
            row["rise_rate_all"], row["decay_rate_all"] = self._max_rates(dt, flux)

            # общие статистики
            for k, v in self._safe_stats(pd.Series(flux)).items():
                row[f"flux_all_{k}"] = v
            row.update({f"flux_all_{k}": v for k, v in self._weighted_stats(flux, ferr).items()})

            # SNR
            snr = np.where(ferr > 0, flux / ferr, np.nan)
            for k, v in self._safe_stats(pd.Series(snr)).items():
                row[f"snr_all_{k}"] = v

            # time to peak + decay fit
            if len(dt) > 0:
                peak_idx = np.nanargmax(flux)
                t_peak = dt[peak_idx]
                row["t_peak_all"] = t_peak
                row["decay_time_all"] = np.nanmax(dt) - t_peak
                row["peak_flux"] = float(flux[peak_idx])
                row["peak_snr"] = float(flux[peak_idx] / ferr[peak_idx]) if ferr[peak_idx] > 0 else np.nan
                row.update(self._decay_fit(dt, flux, t_peak))
                row.update(self._shape_metrics(dt, flux))
            else:
                row["t_peak_all"] = np.nan
                row["decay_time_all"] = np.nan
                row["peak_flux"] = np.nan
                row["peak_snr"] = np.nan
                row.update({"decay_slope": np.nan, "decay_r2": np.nan, "decay_n": 0})

            # normalized flux (per object)
            med = np.nanmedian(flux)
            mad = np.nanmedian(np.abs(flux - med))
            scale = mad if mad > 0 else np.nanstd(flux)
            if scale == 0 or not np.isfinite(scale):
                flux_z = np.full_like(flux, np.nan, dtype=float)
            else:
                flux_z = (flux - med) / scale
            for k, v in self._safe_stats(pd.Series(flux_z)).items():
                row[f"fluxz_all_{k}"] = v

            # time-binned stats
            row.update(self._bin_stats(dt, flux))

            # по фильтрам
            for f in FILTERS:
                sub = grp[grp["Filter"] == f]
                sub_flux = sub["Flux"]
                sub_dt = sub["dt"].to_numpy()
                sub_err = sub["Flux_err"].to_numpy()

                fs = self._safe_stats(sub_flux)
                for k, v in fs.items():
                    row[f"flux_{f}_{k}"] = v
                row.update({f"flux_{f}_{k}": v for k, v in self._weighted_stats(sub_flux.to_numpy(), sub_err).items()})
                row[f"flux_{f}_slope"] = self._slope(sub_dt, sub_flux.to_numpy())
                rr, dr = self._max_rates(sub_dt, sub_flux.to_numpy())
                row[f"flux_{f}_rise_rate"] = rr
                row[f"flux_{f}_decay_rate"] = dr

                # SNR per filter
                sub_snr = np.where(sub_err > 0, sub_flux.to_numpy() / sub_err, np.nan)
                for k, v in self._safe_stats(pd.Series(sub_snr)).items():
                    row[f"snr_{f}_{k}"] = v

            # цвета (разница средних)
            for a, b in [("g", "r"), ("r", "i"), ("i", "z"), ("u", "g"), ("z", "y")]:
                ma = row.get(f"flux_{a}_mean")
                mb = row.get(f"flux_{b}_mean")
                row[f"color_{a}_{b}"] = ma - mb if pd.notna(ma) and pd.notna(mb) else np.nan

            features.append(row)

        return pd.DataFrame(features)


In [41]:
featurs = Features()


### Номрализация фич и подготовка train / test 

In [42]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_recall_curve
try:
    from catboost import CatBoostClassifier
except ImportError as e:
    raise ImportError('CatBoost not installed. Run: pip install catboost') from e
from sklearn.isotonic import IsotonicRegression


In [43]:
def build_preprocessor():
    imputer = SimpleImputer(strategy='median')
    scaler = StandardScaler()
    return imputer, scaler

def apply_preprocessor(X_train, X_val, imputer, scaler):
    X_train_imp = imputer.fit_transform(X_train)
    X_val_imp = imputer.transform(X_val)
    X_train_sc = scaler.fit_transform(X_train_imp)
    X_val_sc = scaler.transform(X_val_imp)
    return X_train_sc, X_val_sc


In [44]:
# Notebook defaults to avoid undefined-variable warnings
top_features = None
fold_results = []


In [45]:
def make_feature_matrix(df: pd.DataFrame, feature_list=None):
    drop_cols = ["object_id", "target", "SpecType", "English Translation", "split", "Z_err"]
    cols = [c for c in df.columns if c not in drop_cols]
    if feature_list is not None:
        cols = [c for c in cols if c in feature_list]
    X = df[cols].copy()
    X = X.apply(pd.to_numeric, errors="coerce")
    return X


#### Подбор порога по F1

In [46]:
def best_f1_threshold(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    # precision/recall длиной на 1 больше, чем thresholds
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    idx = np.nanargmax(f1)
    return thresholds[idx], f1[idx], precision[idx], recall[idx]


## Обучение модели

In [47]:
def train_one_fold(train_meta, val_meta, train_lc, val_lc):
    train_feats = featurs.build_features(train_lc)
    val_feats = featurs.build_features(val_lc)
    train_df = train_meta.merge(train_feats, on="object_id", how="left")
    val_df = val_meta.merge(val_feats, on="object_id", how="left")

    feature_list = top_features if 'top_features' in globals() else None
    X_train = make_feature_matrix(train_df, feature_list)
    y_train = train_df["target"].astype(int).to_numpy()

    X_val = make_feature_matrix(val_df, feature_list)
    y_val = val_df["target"].astype(int).to_numpy()

    imputer, scaler = build_preprocessor()
    X_train_sc, X_val_sc = apply_preprocessor(X_train, X_val, imputer, scaler)

    pos = y_train.sum()
    neg = len(y_train) - pos
    w_pos = neg / max(pos, 1)
    class_weights = [1.0, float(w_pos)]

    model = CatBoostClassifier(
        iterations=2000,
        depth=6,
        learning_rate=0.03,
        loss_function='Logloss',
        eval_metric='AUC',
        random_seed=42,
        class_weights=class_weights,
        l2_leaf_reg=5,
        rsm=0.8,
        subsample=0.8,
        od_type='Iter',
        od_wait=200,
        verbose=200,
    )

    print('CatBoost params:', model.get_params())
    model.fit(X_train_sc, y_train, eval_set=(X_val_sc, y_val), use_best_model=True)

    val_prob = model.predict_proba(X_val_sc)[:, 1]
    thr, f1, p, r = best_f1_threshold(y_val, val_prob)

    print(f"Best F1={f1:.4f}, thr={thr:.3f}, P={p:.4f}, R={r:.4f}")
    return model, imputer, scaler, thr, val_prob, val_meta["object_id"].values


### Полный Cross Validation цикл по сплитам

In [48]:
all_splits = sorted(train_log["split"].unique())
folds = get_splits(train_log, n_folds=5)

fold_results = []
id_to_idx = {oid: i for i, oid in enumerate(train_log["object_id"].values)}
oof_pred = np.full(len(train_log), np.nan)
oof_true = train_log["target"].astype(int).to_numpy()

for fold_id, val_splits in enumerate(folds):
    print('\nFOLD', fold_id, 'val_splits:', val_splits)

    train_splits = [s for s in all_splits if s not in val_splits]
    train_meta = train_log[train_log["split"].isin(train_splits)].copy()
    val_meta = train_log[train_log["split"].isin(val_splits)].copy()

    train_lc = curve.load_lightcurves_for_splits(train_splits, kind="train")
    val_lc = curve.load_lightcurves_for_splits(val_splits, kind="train")
    train_lc = curve.add_time_features(curve.clean_lightcurves(train_lc))
    val_lc = curve.add_time_features(curve.clean_lightcurves(val_lc))

    model, imputer, scaler, thr, val_prob, val_obj_ids = train_one_fold(train_meta, val_meta, train_lc, val_lc)
    fold_results.append({"fold": fold_id, "thr": thr})

    for oid, p in zip(val_obj_ids, val_prob):
        oof_pred[id_to_idx[oid]] = p

mask = np.isfinite(oof_pred)
if not np.all(mask):
    print('WARNING: missing OOF predictions:', np.sum(~mask))

oof_thr, oof_f1, oof_p, oof_r = best_f1_threshold(oof_true[mask], oof_pred[mask])
print(f'OOF Best F1={oof_f1:.4f}, thr={oof_thr:.3f}, P={oof_p:.4f}, R={oof_r:.4f}')



FOLD 0 val_splits: ['split_01', 'split_06', 'split_11', 'split_16']
CatBoost params: {'iterations': 2000, 'learning_rate': 0.03, 'depth': 6, 'l2_leaf_reg': 5, 'rsm': 0.8, 'loss_function': 'Logloss', 'od_wait': 200, 'od_type': 'Iter', 'random_seed': 42, 'verbose': 200, 'class_weights': [1.0, 20.14782608695652], 'eval_metric': 'AUC', 'subsample': 0.8}
0:	test: 0.8094264	best: 0.8094264 (0)	total: 64ms	remaining: 2m 7s
200:	test: 0.8661529	best: 0.8663102 (199)	total: 1.1s	remaining: 9.89s
400:	test: 0.8658383	best: 0.8717626 (290)	total: 2.21s	remaining: 8.82s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.8717626088
bestIteration = 290

Shrink model to first 291 iterations.
Best F1=0.3505, thr=0.322, P=0.2656, R=0.5152

FOLD 1 val_splits: ['split_02', 'split_07', 'split_12', 'split_17']
CatBoost params: {'iterations': 2000, 'learning_rate': 0.03, 'depth': 6, 'l2_leaf_reg': 5, 'rsm': 0.8, 'loss_function': 'Logloss', 'od_wait': 200, 'od_type': 'Iter', 'random_seed':

### Финальный порог

In [49]:
# финальный порог: предпочтительно OOF, иначе медиана фолдов
if 'oof_thr' in globals():
    final_thr = float(oof_thr)
else:
    fold_thresholds = [r["thr"] for r in fold_results]
    final_thr = float(np.median(fold_thresholds))
print('Final threshold:', final_thr)


Final threshold: 0.4050686760673804


### final model

In [50]:
def train_full_model(train_log):
    all_splits = sorted(train_log["split"].unique())
    train_lc = curve.load_lightcurves_for_splits(all_splits, kind="train")
    train_lc = curve.add_time_features(curve.clean_lightcurves(train_lc))

    train_feats = featurs.build_features(train_lc)
    train_df = train_log.merge(train_feats, on="object_id", how="left")

    feature_list = top_features if 'top_features' in globals() else None
    X_train = make_feature_matrix(train_df, feature_list)
    y_train = train_df["target"].astype(int).to_numpy()

    imputer, scaler = build_preprocessor()
    X_train_sc, _ = apply_preprocessor(X_train, X_train, imputer, scaler)

    pos = y_train.sum()
    neg = len(y_train) - pos
    w_pos = neg / max(pos, 1)
    class_weights = [1.0, float(w_pos)]

    model = CatBoostClassifier(
        iterations=2000,
        depth=6,
        learning_rate=0.03,
        loss_function='Logloss',
        eval_metric='AUC',
        random_seed=42,
        class_weights=class_weights,
        l2_leaf_reg=5,
        rsm=0.8,
        subsample=0.8,
        verbose=False,
    )
    model.fit(X_train_sc, y_train)
    return model, imputer, scaler

def predict_test(model, imputer, scaler, test_log):
    all_splits = sorted(test_log["split"].unique())
    test_lc = curve.load_lightcurves_for_splits(all_splits, kind="test")
    test_lc = curve.add_time_features(curve.clean_lightcurves(test_lc))

    test_feats = featurs.build_features(test_lc)
    test_df = test_log.merge(test_feats, on="object_id", how="left")

    feature_list = top_features if 'top_features' in globals() else None
    X_test = make_feature_matrix(test_df, feature_list)
    X_test_imp = imputer.transform(X_test)
    X_test_sc = scaler.transform(X_test_imp)

    prob = model.predict_proba(X_test_sc)[:, 1]
    return test_df["object_id"].values, prob


In [51]:
# Force threshold from kaggle discussions
FORCED_THR = 0.35


In [52]:
# финальный порог
if 'FORCED_THR' in globals():
    final_thr = float(FORCED_THR)
elif 'oof_thr' in globals():
    final_thr = float(oof_thr)
else:
    fold_thresholds = [r["thr"] for r in fold_results]
    final_thr = float(np.median(fold_thresholds))
print('Final threshold:', final_thr)

model, imputer, scaler = train_full_model(train_log)
object_ids, prob = predict_test(model, imputer, scaler, test_log)

pred = (prob >= final_thr).astype(int)
submission = pd.DataFrame({"object_id": object_ids, "prediction": pred})
submission.to_csv("submission.csv", index=False)
print('Saved submission.csv, positives:', pred.sum())

# отчет по OOF
if 'oof_pred' in globals():
    mask = np.isfinite(oof_pred)
    if np.any(mask):
        oof_thr, oof_f1, oof_p, oof_r = best_f1_threshold(oof_true[mask], oof_pred[mask])
        print(f"OOF summary: F1={oof_f1:.4f}, P={oof_p:.4f}, R={oof_r:.4f}, thr={oof_thr:.3f}")


Final threshold: 0.35
Saved submission.csv, positives: 306
OOF summary: F1=0.4011, P=0.3447, R=0.4797, thr=0.405
